# Exercise 1: create an artificial TFBS dataset

Run the cells from top to bottom. No other project files are needed.

We will create 10,000 positive and 10,000 negative DNA sequences, each 100 nt long.
Each positive will contain one sampled 10-nt motif. Negatives will have no planted motif.
Background bases are sampled independently: A = C = G = T = 25%.
These are sampling probabilities, not exactly 25 bases of each type per sequence.


## 1. Import libraries and choose settings

The random seed makes the result reproducible. Change `ofile` to choose the output CSV path.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

n_positive = 10_000
n_negative = 10_000
sequence_length = 100
seed = 42
ofile = "exercise_1_output/dataset.csv"

bases = np.array(list("ACGT"))
rng = np.random.default_rng(seed)


## 2. Convert the PFM into probabilities

Only the **first 10 columns** of the supplied position frequency matrix (PFM) are used.
The four rows are A, C, G, T. Each column describes one motif position.
Divide each column by its own total to obtain the nucleotide probabilities at that position.
We do not add pseudocounts, so zero frequencies stay zero.


In [ ]:
pfm = np.array([
    [87, 167, 281, 56, 8, 744, 40, 107, 851, 5],
    [291, 145, 49, 800, 903, 13, 528, 433, 11, 0],
    [76, 414, 449, 21, 0, 65, 334, 48, 32, 903],
    [459, 187, 134, 36, 2, 91, 11, 324, 18, 3],
])

motif_length = pfm.shape[1]
column_totals = pfm.sum(axis=0)
motif_probabilities = pfm / column_totals

pd.DataFrame(motif_probabilities, index=bases, columns=range(1, motif_length + 1))


## 3. Generate random background sequences

Create one row per sequence and one column per nucleotide.
With no `p` argument, `rng.choice` gives each of the four bases equal probability.
The first 10,000 rows will become positives; the remaining rows will stay negatives.


In [ ]:
n_sequences = n_positive + n_negative
sequences = rng.choice(bases, size=(n_sequences, sequence_length))

print(sequences.shape)
print("".join(sequences[0]))


## 4. Sample one motif for each positive

At each motif position, sample bases using that PFM column's probabilities.
Each positive receives its own sampled motif, rather than a fixed consensus sequence.


In [ ]:
motifs = np.empty((n_positive, motif_length), dtype="U1")

for position in range(motif_length):
    probabilities = motif_probabilities[:, position]
    motifs[:, position] = rng.choice(bases, size=n_positive, p=probabilities)

print("".join(motifs[0]))


## 5. Plant the motifs into the positives

A 10-nt motif can start at positions 0 through 90 in a 100-nt sequence.
`rng.integers` excludes its upper limit, so we use 91 to include start position 90.
Each motif replaces 10 background bases, keeping the sequence length at 100 nt.
We plant motifs in the supplied orientation only.
Negatives are not modified or filtered; chance motif matches can still occur.


In [ ]:
last_start = sequence_length - motif_length
motif_starts = rng.integers(0, last_start + 1, size=n_positive)

for row in range(n_positive):
    start = motif_starts[row]
    sequences[row, start:start + motif_length] = motifs[row]

print("First motif start:", motif_starts[0])
print("".join(sequences[0]))


## 6. Create the dataset table

Join each row of bases into one DNA string. Label positives as 1 and negatives as 0.


In [ ]:
sequence_strings = []

for sequence in sequences:
    sequence_strings.append("".join(sequence))

labels = [1] * n_positive + [0] * n_negative
dataset = pd.DataFrame({
    "sequence": sequence_strings,
    "label": labels,
})

dataset.head()


## 7. Check and save the dataset

Check the number of rows, class counts, and sequence lengths.
Then save the CSV to the `ofile` path chosen at the beginning.
The 25% base probabilities apply to the background; motif planting can change the final base composition.


In [ ]:
assert len(dataset) == n_sequences
assert (dataset["label"] == 1).sum() == n_positive
assert (dataset["label"] == 0).sum() == n_negative
assert (dataset["sequence"].str.len() == sequence_length).all()

print(dataset["label"].value_counts())

Path(ofile).parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(ofile, index=False)
print("Saved:", ofile)


## Optional: download the CSV from Colab

In Colab, add a code cell below and run:

```python
from google.colab import files
files.download(ofile)
```

This download step is only for Colab. The dataset is already saved by the previous cell.
